# Lab 3: Generador de Imágenes con CrewAI

Sistema multi-agente usando **CrewAI** — una tripulación (crew) de agentes con roles, tareas, y herramientas.

**Arquitectura (con archivos de configuración YAML):**
- `config/agents.yaml` — define los agentes: `writer`, `reviewer`, `image_creator`
- `config/tasks.yaml` — define las tareas: `write_task`, `review_task`, `image_task`
- `crew.py` — clase `@CrewBase` que conecta agentes, tareas y herramientas

**Diferencias con Lab 2 (OpenAI Agents SDK):**
- CrewAI usa Crews (tripulaciones) con Tasks secuenciales o jerárquicas
- Los agentes se configuran en YAML con roles, backstory, y goals explícitos
- Las herramientas se definen con `@tool` y se asignan en la clase Crew
- El flujo se define como `Process.sequential` o `Process.hierarchical`

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

for key in ("OPENAI_API_KEY", "GOOGLE_API_KEY"):
    val = os.getenv(key)
    if val:
        os.environ[key] = val
        print(f"{key}: {val[:8]}...{val[-4:]}")
    else:
        print(f"{key} not found.")

In [ ]:
# Importar la Crew desde el módulo con archivos de configuración
import sys
sys.path.insert(0, "src")

from lab3_crewai.crew import ImageGeneratorCrew, last_image_path, gemini_client, GEMINI_IMAGE_MODEL, IMAGE_GEN_CONFIG
import lab3_crewai.crew as crew_module

print("Estructura del proyecto CrewAI:")
print("  src/lab3_crewai/")
print("  ├── config/")
print("  │   ├── agents.yaml    ← define writer, reviewer, image_creator")
print("  │   └── tasks.yaml     ← define write_task, review_task, image_task")
print("  └── crew.py            ← @CrewBase clase + @tool generate_image")

In [ ]:
# Crear y ejecutar la Crew usando la clase @CrewBase con configs YAML

def run_crew(topic: str):
    """Instancia la Crew y la ejecuta con el tema dado."""
    crew_instance = ImageGeneratorCrew()
    result = crew_instance.crew().kickoff(inputs={"topic": topic})
    return result

print("run_crew() definida — usa ImageGeneratorCrew(@CrewBase) con YAML configs")

In [ ]:
import tempfile
import traceback
from pathlib import Path
from google.genai import types
import gradio as gr

RESTYLE_PROMPT = """
Recreate this image as a 16:9 landscape hand-drawn whiteboard illustration on off-white paper with faint blueprint grid lines.

TEXT RULES:
- Translate ONLY descriptive text and labels to Spanish
- DO NOT translate proper names, brand names, product names — keep them EXACTLY as they appear
- DO NOT translate technical acronyms (API, SDK, HTTP, REST, etc.)

LOGO RULES:
- Leave logos UNTOUCHED — same shape, same original colors, no modifications
- Do NOT add any text near or around logos
- Do NOT recolor logos with watercolor or any other fill
- Do NOT invent text to describe or label what a logo is

Keep the same concepts and relationships, but redraw NON-LOGO elements in this style:
- Hand-drawn pen-and-ink sketch with loose cross-hatching
- Subtle watercolor color accents for boxes and arrows only
- Small decorative doodles: stars, lightbulbs, checkmarks
- Handwritten script for labels
- Analog whiteboard feel. No digital vectors, no title header.
"""


# --- Pestaña 1: Generación con CrewAI ---

def generate_from_topic(message, history):
    try:
        yield "Iniciando Crew de agentes CrewAI (config YAML)..."

        result = run_crew(message)

        current_path = crew_module.last_image_path
        if current_path and Path(current_path).exists():
            yield gr.Image(current_path)
        else:
            yield f"**Resultado de la Crew:**\n{result.raw}"

    except Exception as e:
        yield f"**Error:**\n```\n{type(e).__name__}: {e}\n```\n\n```\n{traceback.format_exc()}\n```"


# --- Pestaña 2: Regenerar imagen subida ---

def restyle_image(message, history):
    try:
        text = message.get("text", "").strip()
        files = message.get("files", [])

        if not files:
            yield "Pega o sube una imagen para regenerar."
            return

        image_path = files[0]
        image_bytes = Path(image_path).read_bytes()

        prompt = RESTYLE_PROMPT
        if text:
            prompt += f"\n\nAdditional instructions: {text}"

        yield "Regenerando imagen en estilo whiteboard 16:9 (Gemini)..."

        response = gemini_client.models.generate_content(
            model=GEMINI_IMAGE_MODEL,
            contents=[
                types.Content(
                    parts=[
                        types.Part.from_bytes(data=image_bytes, mime_type="image/png"),
                        types.Part.from_text(text=prompt),
                    ]
                )
            ],
            config=IMAGE_GEN_CONFIG,
        )

        result_bytes = None
        for part in response.candidates[0].content.parts:
            if part.inline_data and part.inline_data.data:
                result_bytes = part.inline_data.data
                break

        if not result_bytes:
            yield "**Error:** No se generó imagen en la respuesta de Gemini."
            return

        tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
        tmp.write(result_bytes)
        tmp.close()
        crew_module.last_image_path = tmp.name

        yield gr.Image(tmp.name)

    except Exception as e:
        yield f"**Error:**\n```\n{type(e).__name__}: {e}\n```\n\n```\n{traceback.format_exc()}\n```"


def download_last_image():
    return crew_module.last_image_path


# --- Interfaz ---

with gr.Blocks() as demo:
    gr.Markdown("# Lab 3: Generador de Imágenes con CrewAI")
    gr.Markdown("Tripulación de agentes con **roles**, **tareas**, y **herramientas** (config YAML) | Texto: OpenAI | Imagen: Gemini")

    with gr.Tabs():
        with gr.TabItem("Generar desde tema"):
            gr.ChatInterface(
                fn=generate_from_topic,
                examples=["How multi-agent systems coordinate tasks using handoffs"],
                multimodal=False,
            )

        with gr.TabItem("Regenerar imagen"):
            gr.Markdown("Pega (Ctrl+V) o sube una imagen y se regenerará en estilo whiteboard con textos en español.")
            gr.ChatInterface(
                fn=restyle_image,
                multimodal=True,
            )

    download_btn = gr.DownloadButton("Descargar última imagen", variant="primary")
    download_btn.click(fn=download_last_image, outputs=download_btn)

demo.launch(inline=True)